In [1]:
!pip install plotly pandas numpy

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Libraries ready!")

Libraries ready!


In [7]:
import pandas as pd

df = pd.read_csv('netflix_titles.csv')

print("=" * 55)
print("CHAPTER 1: INTRODUCING THE NETFLIX DATASET")
print("=" * 55)
print(f"""
Netflix is one of the world's largest streaming platforms.
This dataset contains information about Movies and TV Shows
available on Netflix as of mid-2021.

Dataset Shape   : {df.shape[0]} rows × {df.shape[1]} columns
Columns         : {list(df.columns)}
""")
df.head()

CHAPTER 1: INTRODUCING THE NETFLIX DATASET

Netflix is one of the world's largest streaming platforms.
This dataset contains information about Movies and TV Shows
available on Netflix as of mid-2021.

Dataset Shape   : 8807 rows × 12 columns
Columns         : ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']



,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [8]:
print("Cleaning data before analysis...\n")

df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna(df['rating'].mode()[0], inplace=True)
df['duration'].fillna('Unknown', inplace=True)
df.dropna(subset=['date_added'], inplace=True)

if df['date_added'].dtype == object:
    df['date_added'] = df['date_added'].str.strip()
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df.dropna(subset=['date_added'], inplace=True)

df['year_added']    = df['date_added'].dt.year
df['month_added']   = df['date_added'].dt.month_name()
df['month_num']     = df['date_added'].dt.month
df['duration_int']  = df['duration'].str.extract(r'(\d+)').astype(float)
df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

print(f"Clean dataset ready! {df.shape[0]} records remaining.")

Cleaning data before analysis...

Clean dataset ready! 8797 records remaining.


In [9]:
print("=" * 55)
print("CHAPTER 2: EXPLORATORY DATA ANALYSIS")
print("=" * 55)

print("\nBasic Statistics:")
print(df.describe())

print("\nContent Type Breakdown:")
print(df['type'].value_counts())

print("\nTop 5 Countries:")
print(df['country'].value_counts().head())

print("\nTop 5 Genres:")
print(df['primary_genre'].value_counts().head())

print("\nContent Added Per Year:")
print(df['year_added'].value_counts().sort_index())

CHAPTER 2: EXPLORATORY DATA ANALYSIS

Basic Statistics:
                          date_added  release_year   year_added    month_num  \
count                           8797   8797.000000  8797.000000  8797.000000   
mean   2019-05-17 05:59:08.436966912   2014.183472  2018.871888     6.654996   
min              2008-01-01 00:00:00   1925.000000  2008.000000     1.000000   
25%              2018-04-06 00:00:00   2013.000000  2018.000000     4.000000   
50%              2019-07-02 00:00:00   2017.000000  2019.000000     7.000000   
75%              2020-08-19 00:00:00   2019.000000  2020.000000    10.000000   
max              2021-09-25 00:00:00   2021.000000  2021.000000    12.000000   
std                              NaN      8.822191     1.574243     3.436554   

       duration_int  
count   8794.000000  
mean      69.920173  
min        1.000000  
25%        2.000000  
50%       88.000000  
75%      106.000000  
max      312.000000  
std       50.797005  

Content Type Breakdown:


In [10]:
print("STORY 1: Netflix's Content Growth Journey")

yearly = df.groupby(['year_added','type']).size().reset_index(name='Count')

fig1 = px.area(yearly, x='year_added', y='Count', color='type',
               title='Story 1: How Netflix Grew Its Content Library Over the Years',
               color_discrete_sequence=['#E50914','#B81D24'],
               labels={'year_added':'Year','Count':'Number of Titles'})
fig1.update_layout(template='plotly_dark',
                   annotations=[dict(x=2019, y=1500,
                   text="🚀 Peak content addition!",
                   showarrow=True, arrowhead=2,
                   font=dict(color='white'))])
fig1.show()

print("""
   Insight: Netflix massively increased content additions between
   2015–2019, showing aggressive global expansion.
   After 2019, growth slowed — possibly due to COVID-19 production delays.
""")

STORY 1: Netflix's Content Growth Journey



   Insight: Netflix massively increased content additions between
   2015–2019, showing aggressive global expansion.
   After 2019, growth slowed — possibly due to COVID-19 production delays.



In [11]:
print("STORY 2: Movies vs TV Shows — Who Dominates?")

type_year = df.groupby(['year_added','type']).size().unstack(fill_value=0).reset_index()

fig2 = px.bar(df, x='type', color='type',
              title='Story 2: Movies vs TV Shows — The Content War',
              color_discrete_sequence=['#E50914','#564d4d'],
              labels={'type':'Content Type','count':'Number of Titles'})
fig2.update_layout(template='plotly_dark', showlegend=False)
fig2.show()

print("""
  Insight: Movies outnumber TV Shows nearly 2:1 on Netflix.
   However, TV Shows drive more watch-hours due to multiple episodes.
   Netflix has been investing more in original TV series recently.
""")

STORY 2: Movies vs TV Shows — Who Dominates?



  Insight: Movies outnumber TV Shows nearly 2:1 on Netflix.
   However, TV Shows drive more watch-hours due to multiple episodes.
   Netflix has been investing more in original TV series recently.



In [12]:
print("STORY 3: The Global Content Map")

top_countries = df[df['country'] != 'Unknown']['country'].value_counts().head(15).reset_index()
top_countries.columns = ['Country','Count']

fig3 = px.choropleth(top_countries,
                     locations='Country',
                     locationmode='country names',
                     color='Count',
                     title='Story 3: Where Does Netflix Content Come From?',
                     color_continuous_scale='Reds')
fig3.update_layout(template='plotly_dark')
fig3.show()

print("""
   Insight: The United States dominates Netflix content production,
   followed by India and the UK. This shows Netflix's strong focus
   on English-language and Bollywood content for global audiences.
""")

STORY 3: The Global Content Map



   Insight: The United States dominates Netflix content production,
   followed by India and the UK. This shows Netflix's strong focus
   on English-language and Bollywood content for global audiences.



In [13]:
print("STORY 4: The Genre Landscape")

genre_counts = df['primary_genre'].value_counts().head(12).reset_index()
genre_counts.columns = ['Genre','Count']

fig4 = px.treemap(genre_counts, path=['Genre'], values='Count',
                  title='Story 4: Genre Landscape on Netflix',
                  color='Count', color_continuous_scale='Reds')
fig4.update_layout(template='plotly_dark')
fig4.show()

print("""
   Insight: International Movies and Dramas dominate Netflix's catalog.
   This reflects Netflix's heavy investment in non-English content
   to capture global markets — especially Asia and Europe.
""")

STORY 4: The Genre Landscape



   Insight: International Movies and Dramas dominate Netflix's catalog.
   This reflects Netflix's heavy investment in non-English content
   to capture global markets — especially Asia and Europe.



In [14]:
print("STORY 5: Netflix's Content Calendar")

month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

monthly = df.groupby('month_added').size().reset_index(name='Count')
monthly['month_added'] = pd.Categorical(monthly['month_added'],
                                         categories=month_order, ordered=True)
monthly = monthly.sort_values('month_added')

fig5 = px.bar(monthly, x='month_added', y='Count',
              title='Story 5: Which Month Does Netflix Add the Most Content?',
              color='Count', color_continuous_scale='Reds',
              text='Count')
fig5.update_layout(template='plotly_dark', xaxis_tickangle=-30)
fig5.show()

print("""
   Insight: Netflix adds the most content in January and July —
   aligning with New Year resolutions and summer binge-watching seasons.
   This is a deliberate strategy to maximize subscriber engagement.
""")

STORY 5: Netflix's Content Calendar



   Insight: Netflix adds the most content in January and July —
   aligning with New Year resolutions and summer binge-watching seasons.
   This is a deliberate strategy to maximize subscriber engagement.



In [15]:
print("=" * 55)
print("CHAPTER 5: FINAL CONCLUSIONS & RECOMMENDATIONS")
print("=" * 55)

print("""
📊 KEY FINDINGS FROM OUR DATA STORY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. GROWTH PEAK (2015–2019)
   Netflix added content aggressively during this period,
   nearly tripling its library in just 4 years.

2. MOVIES DOMINATE
   Movies make up ~70% of Netflix content.
   But TV Shows generate higher engagement per title.

3. USA LEADS, INDIA RISING
   The US contributes the most content globally.
   India is the fastest-growing content market on Netflix.

4. INTERNATIONAL CONTENT IS KING
   International Movies & Dramas are the top genres,
   showing Netflix's successful global content strategy.

5. STRATEGIC RELEASE TIMING
   January and July see peak content additions,
   timed to maximize viewer engagement.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 RECOMMENDATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Invest more in TV Show originals — higher retention
Expand Indian & Korean content — fastest growing markets
Focus on International Dramas — highest demand genre
Time major releases in January & July for maximum impact
Reduce content gaps post-2019 to maintain growth momentum
""")

CHAPTER 5: FINAL CONCLUSIONS & RECOMMENDATIONS

📊 KEY FINDINGS FROM OUR DATA STORY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. GROWTH PEAK (2015–2019)
   Netflix added content aggressively during this period,
   nearly tripling its library in just 4 years.

2. MOVIES DOMINATE
   Movies make up ~70% of Netflix content.
   But TV Shows generate higher engagement per title.

3. USA LEADS, INDIA RISING
   The US contributes the most content globally.
   India is the fastest-growing content market on Netflix.

4. INTERNATIONAL CONTENT IS KING
   International Movies & Dramas are the top genres,
   showing Netflix's successful global content strategy.

5. STRATEGIC RELEASE TIMING
   January and July see peak content additions,
   timed to maximize viewer engagement.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 RECOMMENDATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Invest more in TV Show originals — higher retention
Expand Indian & Korean content — fastest growing markets
Focus on International Dramas —

In [16]:
from google.colab import files

with open("netflix_storytelling.html", "w") as f:
    f.write("<h1 style='text-align:center;font-family:Arial;color:#E50914'>📖 Netflix Data Story</h1>")
    f.write("<h3 style='text-align:center;font-family:Arial;color:gray'>A Visual Journey Through Netflix's Content Library</h3>")
    f.write(fig1.to_html(full_html=False))
    f.write(fig2.to_html(full_html=False))
    f.write(fig3.to_html(full_html=False))
    f.write(fig4.to_html(full_html=False))
    f.write(fig5.to_html(full_html=False))

files.download('netflix_storytelling.html')
print("Story app exported!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Story app exported!
